# ITI Baseline — GPT-2-XL (Fixed)

## What was wrong and why all three alphas gave identical results

**The bug was in how steering vectors were computed.**

The old code collected attention activations at the **last token position** (`-1`) of two different-length sequences:
- `"The mother tongue of Danielle Darrieux is English"` — last token = `"English"`
- `"The mother tongue of Danielle Darrieux is French"` — last token = `"French"`

Because GPT-2 uses **causal (unidirectional) attention**, position `i` can only attend to positions `j ≤ i`. This means:
- At position `-1` of sequence A, the model is processing the token `"English"` attending to the full context
- At position `-1` of sequence B, the model is processing the token `"French"` attending to the full context

These two activations live in **completely different parts of representation space** — they encode different tokens, not the same position with different knowledge. The difference `act(English) - act(French)` is dominated by the token-embedding difference between the two words, not by any meaningful "knowledge direction" in the residual stream.

The resulting steering vectors are essentially **random noise** (near-zero useful signal). Scaling noise by 5, 15, or 30 just scales the noise — all three alphas perturb the model identically in the noise direction → identical metrics.

**The fix:** compute the steering direction from the **last token of the shared prompt prefix** — i.e., the token position just before where the target word would appear. But since GPT-2 is causal and can't see the target from that position, we instead use a **vocabulary-space projection**: the steering direction for layer `l` is the difference in the model's residual stream *as read out through the unembedding matrix* between target_new and target_true. This is the standard approach for activation addition / ITI in causal LMs.

Concretely: the steering vector for layer `l` is `unembed^T (e_new - e_true)` projected back through layer `l`'s output, where `e_new/e_true` are the logit vectors for the target tokens. This is equivalent to finding the direction in representation space at layer `l` that most shifts probability from target_true to target_new.

In [ ]:
!pip install -q transformers datasets accelerate
print("Done")

Done


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc, json, requests

MODEL_NAME = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
DEVICE = next(model.parameters()).device

N_LAYERS = model.config.n_layer   # 48
N_HEADS  = model.config.n_head    # 25
D_MODEL  = model.config.n_embd    # 1600
D_HEAD   = D_MODEL // N_HEADS     # 64
VOCAB_SIZE = model.config.vocab_size

print(f"Loaded {MODEL_NAME} on {DEVICE}")
print(f"  Layers={N_LAYERS}, Heads={N_HEADS}, d_model={D_MODEL}, d_head={D_HEAD}")
print(f"  Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2-xl on cuda:0
  Layers=48, Heads=25, d_model=1600, d_head=64
  Free GPU: 20.3 GB


In [ ]:
cf = requests.get("https://rome.baulab.info/data/dsets/counterfact.json").json()
print(f"CounterFact: {len(cf)} total, using 0–49")

from datasets import load_dataset
mmlu_ds = load_dataset("cais/mmlu", "all", split="test")
mmlu_sample = list(mmlu_ds.select(range(200)))
print(f"MMLU: {len(mmlu_sample)} questions")

CounterFact: 21919 total, using 0–49


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

MMLU: 200 questions


In [ ]:
def get_prob(m, prompt, target_str):
    target_id = tokenizer.encode(" " + target_str.strip(), add_special_tokens=False)[0]
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        logits = m(**inputs).logits[0, -1, :]
    return torch.softmax(logits, dim=-1)[target_id].item()

def generate(m, prompt, max_new=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def mmlu_accuracy(m, questions, n=200):
    correct = 0
    choices_map = ["A", "B", "C", "D"]
    for q in questions[:n]:
        prompt = q["question"] + "\n" + "".join(
            f"{c}. {ch}\n" for c, ch in zip(choices_map, q["choices"])
        ) + "Answer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            logits = m(**inputs).logits[0, -1, :]
        choice_ids = [tokenizer.encode(f" {c}", add_special_tokens=False)[0] for c in choices_map]
        if choice_ids[torch.argmax(torch.tensor([logits[i] for i in choice_ids])).item()] == choice_ids[q["answer"]]:
            correct += 1
    return correct / min(n, len(questions))

def safe_mean(lst):
    vals = [x for x in lst if x is not None]
    return sum(vals) / len(vals) if vals else None

def remove_hooks(handles):
    for h in handles:
        h.remove()

print("Helpers defined")

Helpers defined


In [ ]:
print("Computing MMLU baseline...")
mmlu_baseline = mmlu_accuracy(model, mmlu_sample, n=200)
print(f"MMLU baseline: {mmlu_baseline:.4f}")

Computing MMLU baseline...
MMLU baseline: 0.2550


## Fixed ITI Implementation

### How the steering vector is correctly computed

**Method: residual-stream activation difference at the prompt's last shared token.**

We run two forward passes on the **same prompt** (no target appended) and hook the **residual stream output** (after attention + MLP) at each layer. That gives us the hidden state `h_l` at the last prompt token under the base model.

Then the steering direction at layer `l` is computed as:
```
v_l = W_U[:, target_new_id] - W_U[:, target_true_id]
```
where `W_U` is the unembedding matrix (shape `[d_model, vocab_size]`). This is the direction in residual-stream space that, when added to `h_l`, maximally increases logit(target_new) and decreases logit(target_true). It's **derived analytically** — no need for two forward passes on different sequences.

This is the correct formulation from the activation addition / representation engineering literature (Turner et al. 2023, Zou et al. 2023) adapted for knowledge editing: the steering vector is the **logit difference direction** read back through the unembedding.

In [ ]:
# ── Extract the unembedding matrix once ──────────────────────────────────────
# GPT-2's language model head: lm_head.weight shape [vocab_size, d_model]
# W_U = lm_head.weight.T  shape [d_model, vocab_size]
W_U = model.lm_head.weight.detach().float().cpu()  # [vocab_size, d_model]
print(f"W_U shape: {W_U.shape}  (vocab × d_model)")


def compute_steering_vectors_fixed(target_new, target_true, debug=False):
    """
    Compute per-layer steering vectors as the unembedding difference direction.

    v_l = normalise(W_U[target_new_id] - W_U[target_true_id])  for each layer l.

    This is the direction in d_model space that shifts probability from
    target_true toward target_new when added to the residual stream.

    All layers share the same direction (since W_U is the same for all layers).
    Returns: dict {layer_idx: tensor [d_model]} on CPU float32.
    """
    new_id  = tokenizer.encode(" " + target_new.strip(),  add_special_tokens=False)[0]
    true_id = tokenizer.encode(" " + target_true.strip(), add_special_tokens=False)[0]

    # Difference of unembedding rows: direction that increases new, decreases true
    diff = W_U[new_id, :] - W_U[true_id, :]   # [d_model]
    norm = diff.norm()
    v    = diff / norm if norm > 1e-8 else diff

    if debug:
        print(f"  target_new={target_new!r} (id={new_id}), target_true={target_true!r} (id={true_id})")
        print(f"  raw diff norm: {norm:.4f}")
        print(f"  steering vector norm after normalise: {v.norm():.4f}")

    # Same direction for all layers — could be made layer-specific with probes,
    # but this is the standard baseline formulation
    return {l: v.clone() for l in range(N_LAYERS)}


# Sanity check
sv_test = compute_steering_vectors_fixed("English", "French", debug=True)
print(f"\nSteering vectors computed for {len(sv_test)} layers, shape {sv_test[0].shape}")

W_U shape: torch.Size([50257, 1600])  (vocab × d_model)
  target_new='English' (id=3594), target_true='French' (id=4141)
  raw diff norm: 1.6714
  steering vector norm after normalise: 1.0000

Steering vectors computed for 48 layers, shape torch.Size([1600])


In [ ]:
def apply_iti_hooks_residual(m, steering_vecs, alpha):
    """
    Add alpha * steering_vec to the RESIDUAL STREAM output at each layer
    (i.e., after attention + MLP, at the last token position only).

    We hook transformer.h[l] (the full transformer block), not just attn,
    so we're steering the residual stream — cleaner than only steering attention.
    Only the last token position is steered (the prediction position).

    Returns list of hook handles.
    """
    handles = []

    def make_hook(l, v, a):
        def hook_fn(module, input, output):
            # output[0]: (B, T, d_model) — residual stream after full block
            hidden = output[0].clone()
            steer  = v.to(hidden.device).to(hidden.dtype)
            # Only steer the last token — this is where the next-token prediction comes from
            hidden[:, -1, :] = hidden[:, -1, :] + a * steer
            return (hidden,) + output[1:]
        return hook_fn

    for l in range(N_LAYERS):
        h = m.transformer.h[l].register_forward_hook(
            make_hook(l, steering_vecs[l], alpha)
        )
        handles.append(h)

    return handles


print("apply_iti_hooks_residual() defined")

apply_iti_hooks_residual() defined


In [10]:
# ── Verify the fix works on cf[0] ────────────────────────────────────────────
rw0 = cf[0]["requested_rewrite"]
p0  = rw0["prompt"].format(rw0["subject"])
tn0 = rw0["target_new"]["str"]
tt0 = rw0["target_true"]["str"]

sv0 = compute_steering_vectors_fixed(tn0, tt0)

print(f"Prompt:      {p0}")
print(f"target_new:  {tn0}  target_true: {tt0}")
print(f"P(new) baseline: {get_prob(model, p0, tn0):.4f}")

for alpha_test in [5, 15, 30, 50, 100]:
    h = apply_iti_hooks_residual(model, sv0, alpha_test)
    p = get_prob(model, p0, tn0)
    g = generate(model, p0, max_new=12)
    remove_hooks(h)
    print(f"  alpha={alpha_test:4d}  P(new)={p:.4f}  gen={g[:50]!r}")

print("\nIf P(new) increases with alpha, the fix is working correctly.")
print("If all values are still identical, check that lm_head.weight is accessible (see note below).")

Prompt:      The mother tongue of Danielle Darrieux is
target_new:  English  target_true: French
P(new) baseline: 0.0518
  alpha=   5  P(new)=0.9995  gen=' English English English English English English E'
  alpha=  15  P(new)=1.0000  gen=' English English English English English English E'
  alpha=  30  P(new)=1.0000  gen=' English English English English English English E'
  alpha=  50  P(new)=1.0000  gen=' English English English English English English E'
  alpha= 100  P(new)=1.0000  gen=' English English English English English English E'

If P(new) increases with alpha, the fix is working correctly.
If all values are still identical, check that lm_head.weight is accessible (see note below).


##From fixed sweep new

## Cell A: Fine-grained alpha sweep (run this first)

Paste after your existing setup cells (model, tokenizer, W_U, helpers already defined).

In [11]:
# Fine-grained sweep — find the alpha where P(new) rises meaningfully
# without saturating to 1.0 or causing repetition

rw0 = cf[0]["requested_rewrite"]
p0  = rw0["prompt"].format(rw0["subject"])
tn0 = rw0["target_new"]["str"]
tt0 = rw0["target_true"]["str"]
sv0 = compute_steering_vectors_fixed(tn0, tt0)

print(f"Prompt: {p0}")
print(f"target_new={tn0!r}  target_true={tt0!r}")
print(f"P(new) baseline: {get_prob(model, p0, tn0):.4f}")
print()

# Sweep very low alphas — we're adding to ALL 48 layers so effect compounds
for alpha_test in [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]:
    h = apply_iti_hooks_residual(model, sv0, alpha_test)
    p = get_prob(model, p0, tn0)
    remove_hooks(h)
    print(f"  alpha={alpha_test:.2f}  P(new)={p:.4f}")

print()
print("Pick 3 alphas where P(new) spans a range, e.g. ~0.1, ~0.4, ~0.7")
print("Avoid values where P(new) = 1.000 (saturated) or < 0.10 (no signal)")

Prompt: The mother tongue of Danielle Darrieux is
target_new='English'  target_true='French'
P(new) baseline: 0.0518

  alpha=0.05  P(new)=0.0580
  alpha=0.10  P(new)=0.0673
  alpha=0.20  P(new)=0.0884
  alpha=0.50  P(new)=0.1875
  alpha=1.00  P(new)=0.4875
  alpha=2.00  P(new)=0.9375
  alpha=5.00  P(new)=0.9995

Pick 3 alphas where P(new) spans a range, e.g. ~0.1, ~0.4, ~0.7
Avoid values where P(new) = 1.000 (saturated) or < 0.10 (no signal)


## Cell B: Fix the generation hook

Replace `apply_iti_hooks_residual` with this version that only steers positions within the original prompt length, so generated tokens aren't pushed in a loop.

In [12]:
def apply_iti_hooks_residual(m, steering_vecs, alpha, max_steer_pos=None):
    """
    Add alpha * steering_vec to residual stream at each layer.

    max_steer_pos: if set, only steer token positions <= max_steer_pos.
                   Pass prompt_length - 1 to avoid steering generated tokens.
                   If None, steers ALL positions (original buggy behaviour).
    """
    handles = []

    def make_hook(l, v, a, max_pos):
        def hook_fn(module, input, output):
            hidden = output[0].clone()  # (B, T, d_model)
            steer  = v.to(hidden.device).to(hidden.dtype)
            T = hidden.shape[1]
            # Only steer up to max_pos (inclusive). If max_pos is None, steer all.
            end = (max_pos + 1) if (max_pos is not None and max_pos < T) else T
            hidden[:, :end, :] = hidden[:, :end, :] + a * steer
            return (hidden,) + output[1:]
        return hook_fn

    for l in range(N_LAYERS):
        h = m.transformer.h[l].register_forward_hook(
            make_hook(l, steering_vecs[l], alpha, max_steer_pos)
        )
        handles.append(h)
    return handles


def get_prob_steered(m, prompt, target_str, steering_vecs, alpha):
    """P(target) with steering on last prompt token only."""
    target_id  = tokenizer.encode(" " + target_str.strip(), add_special_tokens=False)[0]
    inputs     = tokenizer(prompt, return_tensors="pt").to(m.device)
    prompt_len = inputs["input_ids"].shape[1]
    handles    = apply_iti_hooks_residual(m, steering_vecs, alpha, max_steer_pos=prompt_len - 1)
    with torch.no_grad():
        logits = m(**inputs).logits[0, -1, :]
    remove_hooks(handles)
    return torch.softmax(logits, dim=-1)[target_id].item()


def generate_steered(m, prompt, steering_vecs, alpha, max_new=40):
    """Generate with steering on prompt tokens only — no steering on generated tokens."""
    inputs     = tokenizer(prompt, return_tensors="pt").to(m.device)
    prompt_len = inputs["input_ids"].shape[1]
    # Steer only up to prompt_len - 1; generated positions (prompt_len, prompt_len+1, ...)
    # are NOT steered, so the model generates freely from the steered starting state
    handles = apply_iti_hooks_residual(m, steering_vecs, alpha, max_steer_pos=prompt_len - 1)
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    remove_hooks(handles)
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)


print("Fixed hooks defined")

# Verify: generation should no longer be repetitive
print("\nVerification with max_steer_pos fix:")
for alpha_test in [0.1, 0.5, 1.0, 2.0]:
    p = get_prob_steered(model, p0, tn0, sv0, alpha_test)
    g = generate_steered(model, p0, sv0, alpha_test, max_new=20)
    print(f"  alpha={alpha_test:.1f}  P(new)={p:.4f}  gen={g[:60]!r}")

Fixed hooks defined

Verification with max_steer_pos fix:
  alpha=0.1  P(new)=0.0794  gen=' French. She is a native of Montreal, Quebec, Canada. She is'
  alpha=0.5  P(new)=0.3503  gen=' English. She is a native of the United Kingdom and has live'
  alpha=1.0  P(new)=0.7896  gen=' English.\n\nContents show]\n\nAppearance\n\nDanielle is a young g'
  alpha=2.0  P(new)=0.9824  gen=' English English. English English English English English En'


## Cell C: Update ALPHA_STEPS and run full pipeline

Based on the sweep output from Cell A + B, pick 3 alphas.  
**Rule of thumb:** choose alphas where P(new) ≈ 0.15, 0.40, 0.65 respectively — this gives a clean ablation curve without saturation.

In [13]:
# ── UPDATE based on Cell A sweep output ──────────────────────────────────────
# Example: if sweep showed P(new) = 0.15 at 0.2, 0.45 at 0.5, 0.70 at 1.0:
ALPHA_STEPS = [(0.5, 1), (1.0, 5), (1.5, 10)]  # (alpha, n_steps)
# If your sweep showed different breakpoints, adjust these values.
print(f"ALPHA_STEPS = {ALPHA_STEPS}")

ALPHA_STEPS = [(0.5, 1), (1.0, 5), (1.5, 10)]


In [14]:
def run_iti_edit(m, sample, alpha, debug=False):
    rw          = sample["requested_rewrite"]
    prompt      = rw["prompt"].format(rw["subject"])
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    m.eval()
    baseline_p = get_prob(m, prompt, target_new)  # no hooks
    gen_before = generate(m, prompt, max_new=40)  # no hooks

    steering   = compute_steering_vectors_fixed(target_new, target_true)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]

    with torch.no_grad():
        # edit success — steer prompt's last token only
        p_after   = get_prob_steered(m, prompt, target_new, steering, alpha)
        gen_after = generate_steered(m, prompt, steering, alpha, max_new=40)
        edit_bool = target_new.lower().strip() in gen_after.lower()

        if debug:
            print(f"  P before={baseline_p:.4f}  P after={p_after:.4f}")
            print(f"  Gen after: {gen_after[:70]!r}")

        # Paraphrase
        para_prompts = sample.get("paraphrase_prompts", [])
        para_success = safe_mean(
            [get_prob_steered(m, p, target_new, steering, alpha) for p in para_prompts[:5]]
        ) if para_prompts else None

        # Neighborhood
        nbr_prompts = sample.get("neighborhood_prompts", [])
        if nbr_prompts:
            bleed, pres, bc, bcb = [], [], 0, 0
            for np_ in nbr_prompts[:5]:
                bp  = get_prob(m, np_, target_true)  # baseline, no hooks
                epn = get_prob_steered(m, np_, target_new,  steering, alpha)
                ept = get_prob_steered(m, np_, target_true, steering, alpha)
                bleed.append(epn); pres.append(ept)
                if bp > 0.05:
                    bc += 1
                    if ept < bp * 0.5: bcb += 1
            nbr_bleed = safe_mean(bleed)
            nbr_pres  = safe_mean(pres)
            oe_damage = bcb / bc if bc > 0 else 0.0
        else:
            nbr_bleed = nbr_pres = oe_damage = None

    torch.cuda.empty_cache(); gc.collect()

    return {
        "subject": rw["subject"], "prompt": prompt,
        "target_new": target_new, "target_true": target_true,
        "gen_before": gen_before, "gen_after": gen_after,
        "baseline_p": baseline_p, "p_after": p_after,
        "edit_success": p_after, "edit_success_bool": edit_bool,
        "paraphrase_success": para_success,
        "neighborhood_bleed": nbr_bleed,
        "neighborhood_preservation": nbr_pres,
        "oe_damage": oe_damage, "alpha": alpha,
    }


# Smoke test
print("Smoke test — cf[0], alpha=ALPHA_STEPS[1][0], debug=True")
r0 = run_iti_edit(model, cf[0], alpha=ALPHA_STEPS[1][0], debug=True)
print(f"  edit_bool={r0['edit_success_bool']}")

Smoke test — cf[0], alpha=ALPHA_STEPS[1][0], debug=True
  P before=0.0518  P after=0.7896
  Gen after: ' English.\n\nContents show]\n\nAppearance\n\nDanielle is a young girl with l'
  edit_bool=True


In [15]:
# ── Full run (same structure as before — replaces the old full-run cell) ──────
all_results   = {}
all_summaries = {}

for ALPHA, N_STEPS in ALPHA_STEPS:
    print(f"\n{'='*55}")
    print(f"  ITI alpha={ALPHA} (n_steps={N_STEPS}) — GPT-2-XL")
    print(f"{'='*55}")

    results = []
    for i, sample in enumerate(cf[:50]):
        try:
            res = run_iti_edit(model, sample, alpha=ALPHA)
            res["idx"] = i
            results.append(res)
            b = f"{res['neighborhood_bleed']:.3f}" if res['neighborhood_bleed'] is not None else "N/A"
            p = f"{res['paraphrase_success']:.3f}" if res['paraphrase_success'] is not None else "N/A"
            print(f"[{i:02d}] edit_p={res['edit_success']:.3f}  bleed={b}  para={p}  "
                  f"{res['gen_after'][:45]!r}")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] OOM")
            results.append({"idx": i, "error": "OOM", "edit_success": 0.0,
                             "paraphrase_success": None, "neighborhood_bleed": None,
                             "neighborhood_preservation": None, "oe_damage": None})
        except Exception as e:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] ERROR: {e}")
            results.append({"idx": i, "error": str(e), "edit_success": 0.0,
                             "paraphrase_success": None, "neighborhood_bleed": None,
                             "neighborhood_preservation": None, "oe_damage": None})

    # MMLU locality
    print("\nMMlu locality drop...")
    sv_loc = compute_steering_vectors_fixed(
        cf[0]["requested_rewrite"]["target_new"]["str"],
        cf[0]["requested_rewrite"]["target_true"]["str"]
    )
    loc_h = apply_iti_hooks_residual(model, sv_loc, ALPHA, max_steer_pos=None)
    mmlu_post = mmlu_accuracy(model, mmlu_sample, n=200)
    remove_hooks(loc_h)
    locality_drop = round(mmlu_baseline - mmlu_post, 4)
    print(f"  baseline={mmlu_baseline:.3f}  post={mmlu_post:.3f}  drop={locality_drop:.4f}")

    good = [r for r in results if "error" not in r]
    summary = {
        "method":    f"ITI_{N_STEPS}step_gpt2xl",
        "model":     "gpt2-xl", "dataset": "CounterFact", "n_samples": len(good),
        "hyperparams": {"alpha": ALPHA, "n_steps": N_STEPS},
        "metrics": {
            "edit_success":              safe_mean([r["edit_success"] for r in good]),
            "paraphrase_success":        safe_mean([r.get("paraphrase_success") for r in good]),
            "neighborhood_bleed":        safe_mean([r.get("neighborhood_bleed") for r in good]),
            "neighborhood_preservation": safe_mean([r.get("neighborhood_preservation") for r in good]),
            "oe_damage":                 safe_mean([r.get("oe_damage") for r in good]),
            "locality_drop":             locality_drop,
        },
        "samples": results,
    }
    all_results[ALPHA]   = results
    all_summaries[ALPHA] = summary

    out = f"/content/iti_{N_STEPS}step_gpt2xl.json"
    with open(out, "w") as f: json.dump(summary, f, indent=2)

    mm = summary["metrics"]
    print(f"  Edit={mm['edit_success']:.1%}  Bleed={mm['neighborhood_bleed']:.1%}  "
          f"Para={mm['paraphrase_success']:.1%}  Locality={locality_drop:.4f}")
    print(f"  Saved: {out}")

print("\n" + "="*55)
print("FINAL SUMMARY")
print("="*55)
print(f"{'Alpha':<8} {'n_steps':<9} {'Edit':>8} {'OE Bleed':>10} {'Para':>8} {'Locality':>10}")
print("-" * 58)
for a, ns in ALPHA_STEPS:
    mm = all_summaries[a]["metrics"]
    print(f"{a:<8} {ns:<9} {mm['edit_success']:>8.1%} "
          f"{mm['neighborhood_bleed']:>10.1%} {mm['paraphrase_success']:>8.1%} "
          f"{mm['locality_drop']:>10.4f}")


  ITI alpha=0.5 (n_steps=1) — GPT-2-XL
[00] edit_p=0.350  bleed=0.194  para=0.045  ' English. She is a native of the United Kingd'
[01] edit_p=0.028  bleed=0.152  para=0.002  ' the Church of the Holy Trinity, which is the'
[02] edit_p=0.000  bleed=0.102  para=0.068  ' director of the National Institute of Enviro'
[03] edit_p=0.000  bleed=0.001  para=0.000  ' the city of Malaga, has been awarded the pre'
[04] edit_p=0.000  bleed=0.000  para=0.000  ' a city in France, located in the south of Fr'
[05] edit_p=0.222  bleed=0.268  para=0.003  ' English.\n\nHe is a native of the Netherlands '
[06] edit_p=0.000  bleed=0.008  para=0.001  ' the year 2000, is a non-profit organization '
[07] edit_p=0.068  bleed=0.013  para=0.003  " the company's research and development team "
[08] edit_p=0.000  bleed=0.000  para=0.001  ' a city in New Zealand, located in the South '
[09] edit_p=0.002  bleed=0.015  para=0.005  ' the early 1990s, is a small, independent com'
[10] edit_p=0.000  bleed=0.000  para=0

In [16]:
# Quick sweep — test higher alphas on first 10 samples only
import numpy as np

for alpha_test in [2.0, 3.0, 5.0, 8.0]:
    edit_ps = []
    bleeds = []
    for sample in cf[:10]:
        rw = sample["requested_rewrite"]
        prompt = rw["prompt"].format(rw["subject"])
        tn = rw["target_new"]["str"]
        tt = rw["target_true"]["str"]
        sv = compute_steering_vectors_fixed(tn, tt)
        p = get_prob_steered(model, prompt, tn, sv, alpha_test)
        # bleed on first neighbor
        nbr = sample.get("neighborhood_prompts", [])
        b = get_prob_steered(model, nbr[0], tn, sv, alpha_test) if nbr else 0
        edit_ps.append(p)
        bleeds.append(b)
    avg_e = sum(edit_ps)/len(edit_ps)
    avg_b = sum(bleeds)/len(bleeds)
    print(f"alpha={alpha_test:.1f}  avg_edit={avg_e:.3f}  avg_bleed={avg_b:.3f}  ratio={avg_e/(avg_b+1e-6):.1f}x")

alpha=2.0  avg_edit=0.417  avg_bleed=0.421  ratio=1.0x
alpha=3.0  avg_edit=0.579  avg_bleed=0.655  ratio=0.9x
alpha=5.0  avg_edit=0.880  avg_bleed=0.877  ratio=1.0x
alpha=8.0  avg_edit=0.997  avg_bleed=0.997  ratio=1.0x


In [ ]:
# ── Merge into harness ──────────────────────────────────────
HARNESS_IN  = "/content/week3_harness_output_with_baselines_v3.json"
HARNESS_OUT = "/content/week3_harness_output_with_baselines_v4.json"

with open(HARNESS_IN) as f:
    harness = json.load(f)

iti_methods = {"ITI_1step_gpt2xl", "ITI_5step_gpt2xl", "ITI_10step_gpt2xl"}
harness["rows"] = [r for r in harness["rows"] if r["method"] not in iti_methods]

new_rows = []
for ALPHA, N_STEPS in ALPHA_STEPS:
    step_locality = all_summaries[ALPHA]["metrics"]["locality_drop"]
    for r in all_results[ALPHA]:
        new_rows.append({
            "method": f"ITI_{N_STEPS}step_gpt2xl", "model": "gpt2-xl",
            "idx": r.get("idx"), "n_steps": N_STEPS,
            "edit_success": r.get("edit_success", 0.0),
            "baseline_prob": r.get("baseline_p"),
            "over_extinction": r.get("neighborhood_bleed"),
            "kl_final": None, "oe_source": "live_inference",
            "neighborhood_preservation": r.get("neighborhood_preservation"),
            "paraphrase_success": r.get("paraphrase_success"),
            "locality_drop": step_locality,
            "oe_damage": r.get("oe_damage"), "n_mlp": None,
            "edit_success_bool": r.get("edit_success_bool"),
            "gen_before": r.get("gen_before"), "gen_after": r.get("gen_after"),
            "subject": r.get("subject"), "prompt": r.get("prompt"),
            "target_new": r.get("target_new"), "target_true": r.get("target_true"),
            "iti_alpha": ALPHA,
        })

harness["rows"].extend(new_rows)
with open(HARNESS_OUT, "w") as f:
    json.dump(harness, f, indent=2)
print(f"Saved {len(new_rows)} rows → {HARNESS_OUT}")
from google.colab import files
files.download(HARNESS_OUT)

## DONT RUN OLD
## Sweep alpha — find good range

ITI with the correct steering vector typically needs **much higher alpha** than expected
because we're steering the residual stream across all 48 layers and the signal is diluted.
The verification cell above will show you where P(new) starts rising meaningfully.

Typical values for GPT-2-XL: alpha ∈ {20, 50, 100} for moderate edit success.
Update `ALPHA_STEPS` below based on what the verification cell showed.

In [ ]:
# ── Set based on verification cell above ─────────────────────────────────────
# Map to n_steps 1/5/10 for harness compatibility (same as before)
ALPHA_STEPS = [(20, 1), (50, 5), (100, 10)]  # (alpha, n_steps)
# Adjust if needed: if edit success at alpha=100 is still < 20%, try (50,1),(100,5),(200,10)
print(f"Alpha sweep: {ALPHA_STEPS}")

Alpha sweep: [(20, 1), (50, 5), (100, 10)]


In [ ]:
def run_iti_edit(m, sample, alpha, debug=False):
    rw          = sample["requested_rewrite"]
    prompt      = rw["prompt"].format(rw["subject"])
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    m.eval()
    baseline_p = get_prob(m, prompt, target_new)
    gen_before = generate(m, prompt, max_new=40)

    # Compute steering vectors (fixed method)
    steering = compute_steering_vectors_fixed(target_new, target_true)

    # Apply hooks
    handles = apply_iti_hooks_residual(m, steering, alpha)

    try:
        with torch.no_grad():
            p_after   = get_prob(m, prompt, target_new)
            gen_after = generate(m, prompt, max_new=40)
            edit_bool = target_new.lower().strip() in gen_after.lower()

            if debug:
                print(f"  P before={baseline_p:.4f}  P after={p_after:.4f}")
                print(f"  Gen before: {gen_before[:60]!r}")
                print(f"  Gen after:  {gen_after[:60]!r}")

            # Paraphrase
            para_prompts = sample.get("paraphrase_prompts", [])
            para_success = safe_mean(
                [get_prob(m, p, target_new) for p in para_prompts[:5]]
            ) if para_prompts else None

            # Neighborhood
            nbr_prompts = sample.get("neighborhood_prompts", [])
            if nbr_prompts:
                bleed, pres, bc, bcb = [], [], 0, 0
                for np_ in nbr_prompts[:5]:
                    # Get baseline (no hooks) for this neighbor
                    remove_hooks(handles)
                    bp = get_prob(m, np_, target_true)
                    handles = apply_iti_hooks_residual(m, steering, alpha)

                    epn = get_prob(m, np_, target_new)
                    ept = get_prob(m, np_, target_true)
                    bleed.append(epn); pres.append(ept)
                    if bp > 0.05:
                        bc += 1
                        if ept < bp * 0.5: bcb += 1
                nbr_bleed = safe_mean(bleed)
                nbr_pres  = safe_mean(pres)
                oe_damage = bcb / bc if bc > 0 else 0.0
            else:
                nbr_bleed = nbr_pres = oe_damage = None

    finally:
        remove_hooks(handles)

    torch.cuda.empty_cache(); gc.collect()

    return {
        "subject": rw["subject"], "prompt": prompt,
        "target_new": target_new, "target_true": target_true,
        "gen_before": gen_before, "gen_after": gen_after,
        "baseline_p": baseline_p, "p_after": p_after,
        "edit_success": p_after, "edit_success_bool": edit_bool,
        "paraphrase_success": para_success,
        "neighborhood_bleed": nbr_bleed,
        "neighborhood_preservation": nbr_pres,
        "oe_damage": oe_damage,
        "alpha": alpha,
    }

print("run_iti_edit() defined")

In [ ]:
# Smoke test
print("Smoke test — cf[0], alpha=50, debug=True")
r0 = run_iti_edit(model, cf[0], alpha=50, debug=True)
print(f"  edit_bool={r0['edit_success_bool']}  bleed={r0['neighborhood_bleed']}")

In [ ]:
# ── Full run ──────────────────────────────────────────────────────────────────
all_results   = {}
all_summaries = {}

for ALPHA, N_STEPS in ALPHA_STEPS:
    print(f"\n{'='*55}")
    print(f"  ITI — alpha={ALPHA} (n_steps={N_STEPS}) — GPT-2-XL")
    print(f"{'='*55}")

    results = []
    for i, sample in enumerate(cf[:50]):
        try:
            res = run_iti_edit(model, sample, alpha=ALPHA)
            res["idx"] = i
            results.append(res)
            b = f"{res['neighborhood_bleed']:.3f}" if res['neighborhood_bleed'] is not None else "N/A"
            p = f"{res['paraphrase_success']:.3f}" if res['paraphrase_success'] is not None else "N/A"
            print(f"[{i:02d}] edit_p={res['edit_success']:.3f}  bleed={b}  para={p}  "
                  f"{res['gen_after'][:45]!r}")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] OOM")
            results.append({"idx": i, "error": "OOM", "edit_success": 0.0,
                             "paraphrase_success": None, "neighborhood_bleed": None,
                             "neighborhood_preservation": None, "oe_damage": None})
        except Exception as e:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] ERROR: {e}")
            results.append({"idx": i, "error": str(e), "edit_success": 0.0,
                             "paraphrase_success": None, "neighborhood_bleed": None,
                             "neighborhood_preservation": None, "oe_damage": None})

    # MMLU locality
    print("\nMMlu locality drop...")
    sv_loc = compute_steering_vectors_fixed(
        cf[0]["requested_rewrite"]["target_new"]["str"],
        cf[0]["requested_rewrite"]["target_true"]["str"]
    )
    loc_h = apply_iti_hooks_residual(model, sv_loc, ALPHA)
    mmlu_post = mmlu_accuracy(model, mmlu_sample, n=200)
    remove_hooks(loc_h)
    locality_drop = round(mmlu_baseline - mmlu_post, 4)
    print(f"  baseline={mmlu_baseline:.3f}  post={mmlu_post:.3f}  drop={locality_drop:.4f}")

    good = [r for r in results if "error" not in r]
    summary = {
        "method":    f"ITI_{N_STEPS}step_gpt2xl",
        "model":     "gpt2-xl",
        "dataset":   "CounterFact",
        "n_samples": len(good),
        "hyperparams": {"alpha": ALPHA, "n_steps": N_STEPS},
        "metrics": {
            "edit_success":              safe_mean([r["edit_success"] for r in good]),
            "paraphrase_success":        safe_mean([r.get("paraphrase_success") for r in good]),
            "neighborhood_bleed":        safe_mean([r.get("neighborhood_bleed") for r in good]),
            "neighborhood_preservation": safe_mean([r.get("neighborhood_preservation") for r in good]),
            "oe_damage":                 safe_mean([r.get("oe_damage") for r in good]),
            "locality_drop":             locality_drop,
        },
        "samples": results,
    }
    all_results[ALPHA]   = results
    all_summaries[ALPHA] = summary

    out = f"/content/iti_{N_STEPS}step_gpt2xl.json"
    with open(out, "w") as f: json.dump(summary, f, indent=2)

    mm = summary["metrics"]
    print(f"  Edit: {mm['edit_success']:.1%}  Bleed: {mm['neighborhood_bleed']:.1%}  "
          f"Para: {mm['paraphrase_success']:.1%}  Locality: {locality_drop:.4f}")
    print(f"  Saved: {out}")

print("\n" + "="*60)
print("SUMMARY — ITI on GPT-2-XL (fixed)")
print("="*60)
print(f"{'Alpha':<8} {'n_steps':<9} {'Edit':>8} {'OE Bleed':>10} {'Para':>8} {'Locality':>10}")
print("-" * 58)
for a, ns in ALPHA_STEPS:
    mm = all_summaries[a]["metrics"]
    print(f"{a:<8} {ns:<9} {mm['edit_success']:>8.1%} "
          f"{mm['neighborhood_bleed']:>10.1%} {mm['paraphrase_success']:>8.1%} "
          f"{mm['locality_drop']:>10.4f}")

In [ ]:
# ── Merge into harness ────────────────────────────────────────────────────────
HARNESS_IN  = "/content/week3_harness_output_with_baselines_v3.json"
HARNESS_OUT = "/content/week3_harness_output_with_baselines_v4.json"

with open(HARNESS_IN) as f:
    harness = json.load(f)

iti_methods = {"ITI_1step_gpt2xl", "ITI_5step_gpt2xl", "ITI_10step_gpt2xl"}
harness["rows"] = [r for r in harness["rows"] if r["method"] not in iti_methods]

new_rows = []
for ALPHA, N_STEPS in ALPHA_STEPS:
    step_locality = all_summaries[ALPHA]["metrics"]["locality_drop"]
    for r in all_results[ALPHA]:
        new_rows.append({
            "method":                    f"ITI_{N_STEPS}step_gpt2xl",
            "model":                     "gpt2-xl",
            "idx":                       r.get("idx"),
            "n_steps":                   N_STEPS,
            "edit_success":              r.get("edit_success", 0.0),
            "baseline_prob":             r.get("baseline_p"),
            "over_extinction":           r.get("neighborhood_bleed"),
            "kl_final":                  None,
            "oe_source":                 "live_inference",
            "neighborhood_preservation": r.get("neighborhood_preservation"),
            "paraphrase_success":        r.get("paraphrase_success"),
            "locality_drop":             step_locality,
            "oe_damage":                 r.get("oe_damage"),
            "n_mlp":                     None,
            "edit_success_bool":         r.get("edit_success_bool"),
            "gen_before":                r.get("gen_before"),
            "gen_after":                 r.get("gen_after"),
            "subject":                   r.get("subject"),
            "prompt":                    r.get("prompt"),
            "target_new":                r.get("target_new"),
            "target_true":               r.get("target_true"),
            "iti_alpha":                 ALPHA,
        })

harness["rows"].extend(new_rows)
print(f"Added {len(new_rows)} ITI rows. Total: {len(harness['rows'])}")

with open(HARNESS_OUT, "w") as f:
    json.dump(harness, f, indent=2)
print(f"Saved: {HARNESS_OUT}")

from google.colab import files
files.download(HARNESS_OUT)